# ⚠️ LEGACY PROTOTYPE WARNING
This notebook is an early prototype and writes directly to the OMOP tables using UPDATE.
**Do NOT run this against the data/omop_clinical.duckdb database.**
It circumvents the Blinded Review and Adjudication Governance frameworks introduced in later versions.


In [15]:
import re
import time

import duckdb
import ollama

# Setup
DB_PATH = "../data/omop_clinical.duckdb"
MODEL_NAME = "qwen2.5-coder:7b"
SIMILARITY_THRESHOLD = 0.90 # 90% confidence minimum to accept a match

def get_unique_unmapped_conditions():
    """Fetch ALL unique unmapped clinical conditions from DuckDB."""
    print("🔌 Connecting to DuckDB to fetch unmapped conditions...")
    with duckdb.connect(DB_PATH) as con:
        query = """
            SELECT DISTINCT condition_source_value
            FROM condition_occurrence
            WHERE condition_concept_id = 0
            AND condition_source_value IS NOT NULL
        """
        return [row[0] for row in con.execute(query).fetchall()]

def ai_semantic_normalization(raw_term):
    """Uses local LLM to normalize a messy clinical term into a core standard name."""
    system_prompt = """
    You are an expert Clinical Data Informatician.
    Normalize raw, messy clinical text into a clean, core medical term.
    RULES:
    1. Respond ONLY with the core medical term.
    2. Do NOT include any numeric IDs.
    3. Do NOT include any tags in parentheses like '(disorder)', '(finding)', or '(person)'.
    4. Keep it as short and precise as possible.
    """
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"Raw clinical text: '{raw_term}'"}
            ]
        )
        clean_text = response['message']['content'].strip().strip("'").strip('"')
        # Hardcode fallback to ensure tags are removed even if AI forgets
        clean_text = re.sub(r'\([^)]*\)', '', clean_text).strip()
        return clean_text
    except Exception:
        return None

def find_best_match(con, normalized_term):
    """Finds the best OMOP concept using Jaro-Winkler similarity."""
    search_query = """
        SELECT concept_id, concept_name, domain_id,
               jaro_winkler_similarity(LOWER(concept_name), LOWER(?)) AS score
        FROM concept
        WHERE vocabulary_id = 'SNOMED'
        ORDER BY score DESC
        LIMIT 1
    """
    match = con.execute(search_query, [normalized_term]).fetchone()

    if match and match[3] >= SIMILARITY_THRESHOLD:
        return match
    return None

# EXECUTION BLOCK
print("⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING \n" + "-"*50)

unique_terms = get_unique_unmapped_conditions()

if not unique_terms:
    print("✅ No unmapped conditions found! Database is fully normalized.")
else:
    total_terms = len(unique_terms)
    print(f"⚠️ Found {total_terms} UNIQUE unmapped terms. Starting AI pipeline...\n")

    successful_updates = []

    with duckdb.connect(DB_PATH) as con:
        for i, term in enumerate(unique_terms, 1):
            print(f"[{i}/{total_terms}] Raw: '{term}'")

            # 1. AI Normalization
            ai_term = ai_semantic_normalization(term)
            if not ai_term:
                print("   ❌ AI Failed to respond.")
                print("-" * 40)
                continue

            print(f"   ✨ AI:  '{ai_term}'")

            # 2. Database Fuzzy Match
            start_time = time.time()
            match = find_best_match(con, ai_term)
            db_time = time.time() - start_time

            # 3. Queue for Validation
            if match:
                concept_id, concept_name, domain, score = match
                print(f"   🎯 DB:  '{concept_name}' (ID: {concept_id}) | Score: {score:.2f} | Time: {db_time:.1f}s")
                # Add to our bulk update queue (concept_id goes first because of the UPDATE query structure)
                successful_updates.append((concept_id, term))
            else:
                print(f"   ❌ DB:  No match found above {SIMILARITY_THRESHOLD*100}% confidence.")
            print("-" * 40)

        # 4. The Write-Back (Bulk Update for max performance)
        if successful_updates:
            print(f"\n💾 Writing {len(successful_updates)} standardized concepts back to the database in BULK...")

            update_query = """
                UPDATE condition_occurrence
                SET condition_concept_id = ?
                WHERE condition_source_value = ?
                  AND condition_concept_id = 0
            """
            con.executemany(update_query, successful_updates)
            print("✅ Database successfully updated!")
        else:
            print("\n⚠️ No matches met the confidence threshold. Database was not updated.")

    print(f"\n📊 SUMMARY: Successfully mapped {len(successful_updates)} out of {total_terms} unique terms.")

⚙️ STARTING AI-ASSISTED SEMANTIC MAPPING 
--------------------------------------------------
🔌 Connecting to DuckDB to fetch unmapped conditions...
✅ No unmapped conditions found! Database is fully normalized.


In [13]:
import duckdb

DB_PATH = "../data/omop_clinical.duckdb"

print("🔍 AUDIT: Verifying AI-mapped clinical conditions")
with duckdb.connect(DB_PATH) as con:
    # Fetch records in the Condition table that were successfully mapped (ID != 0)
    query = """
        SELECT condition_source_value, condition_concept_id
        FROM condition_occurrence
        WHERE condition_concept_id != 0
        LIMIT 10
    """

    results = con.execute(query).fetchall()
    for row in results:
        print(f"Raw Source Text: '{row[0]:<30}' ➡️ New SNOMED ID: {row[1]}")

🔍 AUDIT: Verifying AI-mapped clinical conditions
Raw Source Text: 'Sprain of ankle (disorder)    ' ➡️ New SNOMED ID: 81151
Raw Source Text: 'Viral sinusitis (disorder)    ' ➡️ New SNOMED ID: 40481087
Raw Source Text: 'Perennial allergic rhinitis (disorder)' ➡️ New SNOMED ID: 40486433
Raw Source Text: 'Sprain of ankle (disorder)    ' ➡️ New SNOMED ID: 81151
Raw Source Text: 'Viral sinusitis (disorder)    ' ➡️ New SNOMED ID: 40481087
Raw Source Text: 'Sprain of ankle (disorder)    ' ➡️ New SNOMED ID: 81151
Raw Source Text: 'Viral sinusitis (disorder)    ' ➡️ New SNOMED ID: 40481087
Raw Source Text: 'Sprain of ankle (disorder)    ' ➡️ New SNOMED ID: 81151
Raw Source Text: 'Viral sinusitis (disorder)    ' ➡️ New SNOMED ID: 40481087
Raw Source Text: 'Sprain of ankle (disorder)    ' ➡️ New SNOMED ID: 81151
